# 09 - Scene difficulty D and per-slot transition risk R
The two signals the adaptive controller (notebook 10) will act on, computed on every val and test frame and **tuned on val only**.

**Global difficulty** `D = w1 B + w2 M + w3 C + w4 V + w5 O + w6 S` (spec section 10), each component in [0,1], larger = harder:
B brightness (dark), M motion (change since the previous frame), C confidence uncertainty, V vehicle density, O occlusion (overlapping vehicle boxes), S occupancy instability (slots that changed over the last 3 frames).
B and M come from the current image. **C, V, O and S come from the previous frame's YOLO11n result** (feedback): they are never taken from the frame they decide about. A missing component (first frame of a sequence, no detections) is re-normalised away, not set to 0, and its availability is recorded.
*What D should find:* frames where YOLO11n gets slots wrong that YOLO11s gets right (the frames worth escalating). Weights (nonnegative, sum 1, 0.1 grid) maximise the val **capture AUC** of those slots: escalate frames in order of D and measure how fast the fixable errors are covered (0.5 = random order, 1 = perfect).

**Per-slot risk** `R = alpha U + beta Delta + gamma A + delta O` (spec section 11), from what the tracker holds before frame t: U uncertainty of the last score, Delta change between the last two scores, A seconds since the last observation / 30 min, O share of the slot covered by a neighbouring vehicle.
*What R should find:* slots whose last state would be **wrong if reused** at frame t (a skipped frame shows the last state). Weights maximise val ROC AUC. Then the global risk of a frame, **max vs top-k mean** of its slot risks, is compared on val (capture AUC of reuse errors) and the better one is chosen.

Test is reported once, at the frozen val choices, next to equal weights and each single component.
Frames are 5 min (PKLot) / 30 min (CNRPark-EXT) apart (no video), so M is a frame-to-frame scene change at that spacing.

**Inputs (Add Input -> Your Work):** notebook 07 output (`artifacts/occupancy/`: slot scores, cached detections, slot registry) and notebook 03 output (val/test images and `artifacts/splits/`).
**Accelerator:** none (CPU). Internet ON only to clone the repo.
**How to run:** `SMOKE = True` + Run All (a few sequences, ~2 min). Then `SMOKE = False` + **Save Version -> Save & Run All** (reads ~4,900 images).

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Settings and inputs
SMOKE = True                      # True: a few sequences per split. False: every val and test sequence
SMOKE_SEQUENCES = 8
FEEDBACK_MODEL = "yolo11n"        # C, V, O, S come from this model's previous frame (Standard mode)
OTHER_MODEL = "yolo11s"
INSTABILITY_FRAMES = 3            # S: slots that changed over the last 3 processed frames
MAX_AGE_S = 1800.0                # A: age normalised by 30 min (the longest normal frame spacing); the controller's max age is notebook 10
GRID_STEP = 0.1                   # weight grid on the simplex
MAX_GAP_FACTOR = 3.0              # same sequence rule as notebook 08
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import os, json, time, csv, gzip
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
from PIL import Image
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
from src.occupancy import association as asc
from src.temporal import filters as tf
from src.scene_analysis import features as sf
from src.slot_risk import risk as rk
DCFG = load_yaml("configs/dataset.yaml"); OCFG = load_yaml("configs/occupancy.yaml"); CCFG = load_yaml("configs/adaptive_controller.yaml")
FRAC = DCFG["dataset"]["subset_fraction"]

mans = [p for p in yt.find_files(INPUT_ROOT, "split_manifest.csv") if p.parent.name == "splits"]
assert mans, "split_manifest.csv not found: attach notebook 03's output"
ROW_BY_ID = {int(r["image_id"]): r for r in de.read_manifest(mans[0])}
SPLIT_INFO = json.loads((mans[0].parent / "split_info.json").read_text())
EXPORT = yt.find_export_dir(INPUT_ROOT, FRAC)

occ = [p.parent for p in yt.find_files(INPUT_ROOT, "slot_scores_test.csv.gz") if p.parent.name == "occupancy"]
assert occ, "slot_scores_test.csv.gz not found: attach notebook 07's output (full run)"
OCC = occ[0]
st07 = json.loads((OCC / "status.json").read_text())
assert st07["state"] == "COMPLETE", "notebook 07 status is {}".format(st07["state"])
THR, ASSOC = {}, {}
for v in (FEEDBACK_MODEL, OTHER_MODEL):
    rec = json.loads((OCC / "occupancy_record_{}.json".format(v)).read_text())
    assert not rec["smoke_test"] and rec["dataset_split_hash"] == SPLIT_INFO["manifest_hash"], v
    THR[v] = float(rec["chosen_on_val"]["conf_thr"]); ASSOC[v] = rec["chosen_on_val"]
SLOTS = {s["slot_id"]: s for s in json.loads((OCC / "slots.json").read_text())}
REGION = {}                                     # camera -> bounding box of all its slots (where B and M are measured)
for s in SLOTS.values():
    xs, ys = s["polygon"][0::2], s["polygon"][1::2]
    b = REGION.get(s["camera"], [1e9, 1e9, -1e9, -1e9])
    REGION[s["camera"]] = [min(b[0], min(xs)), min(b[1], min(ys)), max(b[2], max(xs)), max(b[3], max(ys))]

def load_dets(v, split):
    z = np.load(str(OCC / "detections" / "{}_{}.npz".format(v, split)))
    return {"dets": z["dets"], "starts": z["starts"], "row": {str(n): i for i, n in enumerate(z["names"])}}
DETS = {s: load_dets(FEEDBACK_MODEL, s) for s in ("val", "test")}

def load_scores(split):
    cols = defaultdict(list)
    with gzip.open(str(OCC / "slot_scores_{}.csv.gz".format(split)), "rt") as f:
        for r in csv.DictReader(f):
            for k in ("image", "camera", "timestamp", "slot_id"):
                cols[k].append(r[k])
            cols["image_id"].append(int(r["image_id"])); cols["gt"].append(int(r["gt_occupied"])); cols["valid"].append(int(r["valid"]))
            cols["sn"].append(float(r["score_" + FEEDBACK_MODEL])); cols["ss"].append(float(r["score_" + OTHER_MODEL]))
    return {k: np.array(x) for k, x in cols.items()}
RAW = {s: load_scores(s) for s in ("val", "test")}
print("notebook 07:", OCC, "| thresholds", THR, "| association", {k: ASSOC[FEEDBACK_MODEL][k] for k in ("method", "overlap_thr", "exclusive")})
print("images:", EXPORT, "| slot rows val/test:", len(RAW["val"]["gt"]), len(RAW["test"]["gt"]))
OUT = Path(WORK) / "artifacts" / "scene_risk"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""
CFG_HASH = config_hash({"controller": CCFG, "occupancy": OCFG, "smoke": SMOKE, "smoke_sequences": SMOKE_SEQUENCES, "fb": FEEDBACK_MODEL,
                        "inst": INSTABILITY_FRAMES, "max_age": MAX_AGE_S, "step": GRID_STEP, "gap": MAX_GAP_FACTOR,
                        "split": SPLIT_INFO["manifest_hash"], "nb07_run": st07["run_id"]})
RUN_ID = new_run_id("scn")
STATUS = OUT / "status.json"
write_status(STATUS, "09_scene_difficulty_and_slot_risk", RUN_ID, "RUNNING", CFG_HASH, "inputs loaded")

In [ ]:
# Replay every sequence in time order: per-frame components (D) and per-slot components (R), causal
def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "")).timestamp()

def rnd(x, n=4):
    return None if x is None else round(float(x), n)

PREP = {}
def slots_of(ids):
    if ids not in PREP:
        PREP[ids] = asc.prepare_slots([SLOTS[s]["polygon"] for s in ids])
    return PREP[ids]

def replay(split):
    o, D = RAW[split], DETS[split]
    names, first, counts = np.unique(o["image"], return_index=True, return_counts=True)
    fr = np.argsort(first); names, first, counts = names[fr], first[fr], counts[fr]
    cam = o["camera"][first]; iid = o["image_id"][first]
    sec = np.array([parse_ts(t) for t in o["timestamp"][first]]); day = np.array([ROW_BY_ID[int(i)]["day"] for i in iid])
    sp = tf.split_sequences(cam, sec, day, MAX_GAP_FACTOR)
    seq_ids = sorted(set(sp["seq"].tolist()))
    if SMOKE:
        seq_ids = [seq_ids[i] for i in np.linspace(0, len(seq_ids) - 1, min(SMOKE_SEQUENCES, len(seq_ids))).astype(int)]
    cam_cols = defaultdict(dict)
    for c, s in zip(o["camera"], o["slot_id"]):
        cam_cols[c].setdefault(s, len(cam_cols[c]))
    frames, slots_rows = [], defaultdict(list)
    t0 = time.time()
    for q in seq_ids:
        fidx = np.nonzero(sp["seq"] == q)[0]; fidx = fidx[np.argsort(sec[fidx], kind="stable")]
        c = cam[fidx[0]]; ncol = len(cam_cols[c])
        prev_gray, prev, hist = None, None, []
        for t, f in enumerate(fidx):
            a, b = first[f], first[f] + counts[f]
            sids = tuple(o["slot_id"][a:b]); valid = o["valid"][a:b].astype(bool); gt = o["gt"][a:b]
            sn, ss = o["sn"][a:b], o["ss"][a:b]
            pn, ps = sn >= THR[FEEDBACK_MODEL], ss >= THR[OTHER_MODEL]
            with Image.open(str(EXPORT / "images" / split / names[f])) as im:
                gray = sf.to_gray_small(im, REGION[c])
            k = D["row"][names[f]]; dets = D["dets"][D["starts"][k]:D["starts"][k + 1]]
            pairs = asc.image_pairs(slots_of(sids), dets)
            occ_slot = rk.slot_occlusion(len(sids), pairs, THR[FEEDBACK_MODEL], float(ASSOC[FEEDBACK_MODEL]["overlap_thr"]))
            detf = sf.detection_features(dets, THR[FEEDBACK_MODEL], len(sids))
            state = np.full(ncol, -1, np.int8)
            cols = np.array([cam_cols[c][s] for s in sids])
            state[cols[valid]] = pn[valid]
            r0 = ROW_BY_ID[int(iid[f])]
            wrong_n, wrong_s = (pn != gt.astype(bool)) & valid, (ps != gt.astype(bool)) & valid
            row = {"split": split, "seq": int(q), "t": t, "image": names[f], "camera": c, "site": r0["site"], "climate": r0.get("climate") or "unknown",
                   "day_type": "flagged" if (r0.get("day_flag") or "").strip() else "normal", "seconds": sec[f],
                   "slots": int(valid.sum()), "brightness": sf.brightness(gray), "motion_raw": sf.motion_raw(gray, prev_gray),
                   "confidence": prev["detf"]["confidence"] if prev else None, "density": prev["detf"]["density"] if prev else None,
                   "occlusion_raw": prev["detf"]["occlusion_raw"] if prev else None,
                   "instability": sf.instability(hist[-(INSTABILITY_FRAMES + 1):]) if hist else None,
                   "n_errors": int(wrong_n.sum()), "s_errors": int(wrong_s.sum()), "fixable": int((wrong_n & ~wrong_s).sum()),
                   "reuse_errors": None}
            if prev is not None:
                ru = 0
                for j, s in enumerate(sids):
                    if not valid[j] or s not in prev["slot"]:
                        continue
                    ps_ = prev["slot"][s]
                    err = int(gt[j] != ps_["pred"]); ru += err
                    slots_rows["frame"].append(len(frames)); slots_rows["slot_id"].append(s)
                    slots_rows["uncertainty"].append(float(rk.uncertainty(ps_["score"], THR[FEEDBACK_MODEL])))
                    slots_rows["delta"].append(abs(ps_["score"] - ps_["score_prev"]) if ps_["score_prev"] is not None else np.nan)
                    slots_rows["age"].append(float(rk.age_component(sec[f] - prev["sec"], MAX_AGE_S)))
                    slots_rows["occlusion"].append(ps_["occ"]); slots_rows["target"].append(err)
                row["reuse_errors"] = ru
            frames.append(row)
            new_slot = {}
            for j, s in enumerate(sids):
                if valid[j]:
                    old = prev["slot"].get(s) if prev else None
                    new_slot[s] = {"score": float(sn[j]), "score_prev": old["score"] if old else None, "pred": int(pn[j]), "occ": float(occ_slot[j])}
            prev = {"detf": detf, "slot": new_slot, "sec": sec[f]}
            prev_gray = gray; hist.append(state)
    print("{:<5} {} sequences, {} frames, {} slot rows for R, {:.0f} s".format(split, len(seq_ids), len(frames), len(slots_rows["target"]), time.time() - t0))
    return frames, {k: np.array(v) for k, v in slots_rows.items()}

try:
    FR, SL = {}, {}
    for split in ("val", "test"):
        FR[split], SL[split] = replay(split)
except Exception as e:
    write_status(STATUS, "09_scene_difficulty_and_slot_risk", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500])
    raise

def col(rows, k):
    return np.array([np.nan if r[k] is None else r[k] for r in rows], float)
SCALES = sf.fit_scales({"motion": col(FR["val"], "motion_raw"), "occlusion": col(FR["val"], "occlusion_raw")})
def components(rows):
    F = np.stack([col(rows, "brightness"), sf.scale(col(rows, "motion_raw"), SCALES["motion"]), col(rows, "confidence"),
                  col(rows, "density"), sf.scale(col(rows, "occlusion_raw"), SCALES["occlusion"]), col(rows, "instability")], 1)
    return F
F = {s: components(FR[s]) for s in FR}
print("val-fitted scales (99th percentile):", {k: round(v, 4) for k, v in SCALES.items()})
print("availability (share of frames with the component):")
for s in F:
    print("  {:<5}".format(s), {sf.SHORT[f]: round(float((~np.isnan(F[s][:, i])).mean()), 3) for i, f in enumerate(sf.FEATURES)})
for s in FR:
    n = np.array([r["n_errors"] for r in FR[s]]); fx = np.array([r["fixable"] for r in FR[s]])
    print("  {:<5} YOLO11n slot errors {} | fixable by YOLO11s {} ({:.0%} of n errors) in {} of {} frames".format(
        s, n.sum(), fx.sum(), fx.sum() / max(1, n.sum()), int((fx > 0).sum()), len(fx)))

In [ ]:
# D: choose the weights on VAL (capture AUC of slots YOLO11n gets wrong and YOLO11s gets right)
GRID6 = sf.simplex_grid(len(sf.FEATURES), GRID_STEP)
def d_target(split):
    return np.array([r["fixable"] for r in FR[split]], float)
t0 = time.time()
S_all, _ = sf.weighted_score(F["val"], GRID6)
AUC6 = sf.capture_auc_many(S_all, d_target("val"))
bi = int(np.nanargmax(AUC6)) if not np.isnan(AUC6).all() else None
W_D = GRID6[bi] if bi is not None else np.full(len(sf.FEATURES), 1.0 / len(sf.FEATURES))
EQUAL6 = np.full(len(sf.FEATURES), 1.0 / len(sf.FEATURES))
def d_report(split, w):
    s, _ = sf.weighted_score(F[split], w)
    tgt = d_target(split)
    return {"capture_auc": rnd(sf.capture_auc(s, tgt), 4), **sf.capture_at(s, tgt), "spearman_vs_fixable": rnd(sf.spearman(s, tgt)),
            "spearman_vs_n_errors": rnd(sf.spearman(s, np.array([r["n_errors"] for r in FR[split]], float)))}
D_ROWS = []
for name, w in [("tuned", W_D), ("equal", EQUAL6)] + [("only " + sf.SHORT[f], np.eye(len(sf.FEATURES))[i]) for i, f in enumerate(sf.FEATURES)]:
    D_ROWS.append({"weights": name, **{sf.SHORT[f]: round(float(x), 2) for f, x in zip(sf.FEATURES, w)}, **{"val_" + k: v for k, v in d_report("val", w).items()}})
print("{} weight vectors scored in {:.0f} s; chosen (val capture AUC {:.4f}): {}".format(
    len(GRID6), time.time() - t0, AUC6[bi] if bi is not None else float('nan'), {sf.SHORT[f]: round(float(x), 2) for f, x in zip(sf.FEATURES, W_D)}))
print("top 5 on val:", [({sf.SHORT[f]: round(float(x), 1) for f, x in zip(sf.FEATURES, GRID6[i]) if x > 0}, round(float(AUC6[i]), 4))
                         for i in np.argsort(-np.nan_to_num(AUC6, nan=-1))[:5]])
print()
print("{:<10} {:>4} {:>4} {:>4} {:>4} {:>4} {:>4} {:>9} {:>7} {:>7} {:>7} {:>9}".format("weights", "B", "M", "C", "V", "O", "S", "capAUC", "top10%", "top20%", "top30%", "spearman"))
for r in D_ROWS:
    print("{:<10} {:>4} {:>4} {:>4} {:>4} {:>4} {:>4} {:>9} {:>7} {:>7} {:>7} {:>9}".format(*[str(r[k]) for k in ("weights", "B", "M", "C", "V", "O", "S",
          "val_capture_auc", "val_top10%", "val_top20%", "val_top30%", "val_spearman_vs_fixable")]))

In [ ]:
# R: choose alpha..delta on VAL (slot-level ROC AUC of 'reusing the last state would be wrong'), then max vs top-k mean
GRID4 = sf.simplex_grid(len(rk.COMPONENTS), GRID_STEP)
def r_matrix(split):
    return np.stack([SL[split][c] for c in rk.COMPONENTS], 1).astype(float)
RV, YV = r_matrix("val"), SL["val"]["target"]
t0 = time.time()
RS, _ = sf.weighted_score(RV, GRID4)
AUC4 = np.array([rk.roc_auc(RS[:, j], YV) or np.nan for j in range(len(GRID4))])
ri = int(np.nanargmax(AUC4)) if not np.isnan(AUC4).all() else None
W_R = GRID4[ri] if ri is not None else np.full(len(rk.COMPONENTS), 0.25)
EQUAL4 = np.full(len(rk.COMPONENTS), 0.25)
R_ROWS = []
for name, w in [("tuned", W_R), ("equal", EQUAL4)] + [("only " + rk.SHORT[c], np.eye(4)[i]) for i, c in enumerate(rk.COMPONENTS)]:
    s, _ = sf.weighted_score(RV, w)
    R_ROWS.append({"weights": name, **{rk.SHORT[c]: round(float(x), 2) for c, x in zip(rk.COMPONENTS, w)}, "val_slot_auc": rnd(rk.roc_auc(s, YV))})
print("{} weight vectors in {:.0f} s | val slots {} | reuse-error rate {:.2%}".format(len(GRID4), time.time() - t0, len(YV), YV.mean()))
print("{:<10} {:>4} {:>6} {:>4} {:>4} {:>9}".format("weights", "U", "Delta", "A", "O", "slot AUC"))
for r in R_ROWS:
    print("{:<10} {:>4} {:>6} {:>4} {:>4} {:>9}".format(*[str(r[k]) for k in ("weights", "U", "Delta", "A", "O", "val_slot_auc")]))

def frame_risk(split, w, agg):
    s, _ = sf.weighted_score(r_matrix(split), w)
    fr = SL[split]["frame"]
    out = np.full(len(FR[split]), np.nan)
    by = defaultdict(list)
    for i, f in enumerate(fr):
        by[f].append(s[i])
    for f, v in by.items():
        out[f] = rk.aggregate(np.array(v), agg["method"], agg.get("k"))
    return out
def reuse_target(split):
    return np.array([np.nan if r["reuse_errors"] is None else r["reuse_errors"] for r in FR[split]], float)
AGG_ROWS = []
for agg in rk.aggregation_options():
    g = frame_risk("val", W_R, agg); tgt = reuse_target("val"); m = ~np.isnan(tgt)
    AGG_ROWS.append({"aggregation": agg["method"] + ("" if "k" not in agg else " k={}".format(agg["k"])), **agg,
                     "val_capture_auc": rnd(sf.capture_auc(g[m], tgt[m]), 4), **{"val_" + k: v for k, v in sf.capture_at(g[m], tgt[m]).items()}})
AGG = max(AGG_ROWS, key=lambda r: (r["val_capture_auc"] or 0, r["method"] == "max"))
print()
print("global risk per frame (val capture AUC of reuse errors):")
for r in AGG_ROWS:
    print("  {:<14} capAUC {:<7} top10% {:<7} top20% {:<7}{}".format(*[str(r[k]) for k in ("aggregation", "val_capture_auc", "val_top10%", "val_top20%")], "   <- chosen" if r is AGG else ""))
CHOSEN_AGG = {k: AGG[k] for k in ("method", "k") if k in AGG}

In [ ]:
# TEST at the frozen val choices, overall and per group
TEST = {"D": {}, "R": {}, "aggregation": {}}
for name, w in [("tuned", W_D), ("equal", EQUAL6)] + [("only " + sf.SHORT[f], np.eye(6)[i]) for i, f in enumerate(sf.FEATURES)]:
    TEST["D"][name] = d_report("test", w)
RT, YT = r_matrix("test"), SL["test"]["target"]
for name, w in [("tuned", W_R), ("equal", EQUAL4)] + [("only " + rk.SHORT[c], np.eye(4)[i]) for i, c in enumerate(rk.COMPONENTS)]:
    s, _ = sf.weighted_score(RT, w); TEST["R"][name] = rnd(rk.roc_auc(s, YT))
for agg in rk.aggregation_options():
    g = frame_risk("test", W_R, agg); tgt = reuse_target("test"); m = ~np.isnan(tgt)
    TEST["aggregation"][agg["method"] + ("" if "k" not in agg else " k={}".format(agg["k"]))] = rnd(sf.capture_auc(g[m], tgt[m]), 4)
# does global risk also find the frames worth escalating, and does D find reuse errors? (cross-check, not tuned)
g_test = frame_risk("test", W_R, CHOSEN_AGG); d_test, _ = sf.weighted_score(F["test"], W_D)
TEST["cross"] = {"global_R_vs_fixable_capture_auc": rnd(sf.capture_auc(np.nan_to_num(g_test, nan=np.nanmedian(g_test)), d_target("test")), 4),
                 "D_vs_reuse_errors_capture_auc": rnd(sf.capture_auc(d_test[~np.isnan(reuse_target("test"))], reuse_target("test")[~np.isnan(reuse_target("test"))]), 4)}
print("TEST | D capture AUC of fixable slots (0.5 = random):")
for name, r in TEST["D"].items():
    print("  {:<8} capAUC {:<7} top10% {:<7} top20% {:<7} top30% {:<7} spearman {}".format(name, *[str(r[k]) for k in ("capture_auc", "top10%", "top20%", "top30%", "spearman_vs_fixable")]))
print("TEST | R slot ROC AUC of reuse errors:", TEST["R"])
print("TEST | global risk capture AUC of reuse errors:", TEST["aggregation"], "| chosen:", AGG["aggregation"])
print("TEST | cross-checks:", TEST["cross"])

# per group (frames of the test split)
GROUP_ROWS = []
keys = lambda r: ["site:" + r["site"], "climate:" + r["climate"], "day_type:" + r["day_type"]]
groups = defaultdict(list)
for i, r in enumerate(FR["test"]):
    for g in keys(r):
        groups[g].append(i)
for g, idx in sorted(groups.items()):
    idx = np.array(idx); tgt = d_target("test")[idx]
    ds, _ = sf.weighted_score(F["test"][idx], W_D); de_, _ = sf.weighted_score(F["test"][idx], EQUAL6)
    GROUP_ROWS.append({"group": g, "frames": len(idx), "fixable": int(tgt.sum()), "mean_D": round(float(np.nanmean(ds)), 3),
                       "capture_auc_tuned": None if tgt.sum() == 0 else rnd(sf.capture_auc(ds, tgt), 4),
                       "capture_auc_equal": None if tgt.sum() == 0 else rnd(sf.capture_auc(de_, tgt), 4),
                       **{"mean_" + sf.SHORT[f]: round(float(np.nanmean(F["test"][idx, i])), 3) for i, f in enumerate(sf.FEATURES)}})
print()
print("{:<26} {:>6} {:>8} {:>7} {:>8} {:>8}  {}".format("group", "frames", "fixable", "mean D", "capAUC", "equal", "mean B/M/C/V/O/S"))
for r in GROUP_ROWS:
    print("{:<26} {:>6} {:>8} {:>7} {:>8} {:>8}  {}".format(r["group"], r["frames"], r["fixable"], r["mean_D"], str(r["capture_auc_tuned"]), str(r["capture_auc_equal"]),
          "/".join(str(r["mean_" + sf.SHORT[f]]) for f in sf.FEATURES)))

RECORD = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "dataset_split_hash": SPLIT_INFO["manifest_hash"],
          "notebook07_run_id": st07["run_id"], "feedback_model": FEEDBACK_MODEL, "score_thresholds": THR,
          "difficulty": {"features": list(sf.FEATURES), "weights": {f: round(float(x), 3) for f, x in zip(sf.FEATURES, W_D)},
                         "scales": SCALES, "objective": "val capture AUC of slots YOLO11n wrong and YOLO11s right",
                         "val": D_ROWS, "test": TEST["D"], "groups_test": GROUP_ROWS},
          "slot_risk": {"components": list(rk.COMPONENTS), "weights": {c: round(float(x), 3) for c, x in zip(rk.COMPONENTS, W_R)},
                        "max_age_s_for_A": MAX_AGE_S, "objective": "val slot ROC AUC of reuse errors", "val": R_ROWS, "test": TEST["R"],
                        "aggregation": CHOSEN_AGG, "aggregation_val": AGG_ROWS, "aggregation_test": TEST["aggregation"]},
          "cross_checks_test": TEST["cross"], "instability_frames": INSTABILITY_FRAMES, "grid_step": GRID_STEP,
          "timing": "B, M from the current image; C, V, O, S and every R component from the previous processed frame",
          "limitation": "frames 5 min (PKLot) / 30 min (CNRPark-EXT) apart; A is nearly constant per camera when every frame is processed"}
(OUT / "scene_risk_record{}.json".format(SUF)).write_text(json.dumps(RECORD, indent=2, default=str))
for s in FR:   # per-frame features for notebook 10 (saves re-reading the images)
    with gzip.open(str(OUT / "frame_features_{}{}.csv.gz".format(s, SUF)), "wt", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(FR[s][0])); w.writeheader(); w.writerows(FR[s])

In [ ]:
# Plots: capture curves, D components over one test sequence, one slot's risk over time
import matplotlib.pyplot as plt
def curve(score, tgt):
    s = np.nan_to_num(np.asarray(score, float), nan=np.nanmedian(score)); o = np.argsort(-s, kind="stable")
    return np.arange(1, len(s) + 1) / len(s), np.cumsum(tgt[o]) / max(1e-9, tgt.sum())
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a, split in zip(ax, ("val", "test")):
    tgt = d_target(split)
    for name, w, st in (("D tuned", W_D, "-"), ("D equal", EQUAL6, "--")):
        s, _ = sf.weighted_score(F[split], w); a.plot(*curve(s, tgt), st, label=name)
    g = frame_risk(split, W_R, CHOSEN_AGG); a.plot(*curve(g, tgt), ":", label="global R ({})".format(AGG["aggregation"]))
    a.plot([0, 1], [0, 1], color="0.6", lw=0.8, label="random")
    a.set(xlabel="share of frames escalated (highest score first)", ylabel="share of fixable slots captured", title="{}: frames worth YOLO11s".format(split.upper()))
    a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(str(OUT / "capture_curves{}.png".format(SUF)), dpi=110); plt.show()

# D and its components over the test sequence with the most fixable slots
seqs = defaultdict(list)
for i, r in enumerate(FR["test"]):
    seqs[r["seq"]].append(i)
q = max(seqs, key=lambda q: sum(FR["test"][i]["fixable"] for i in seqs[q]))
idx = np.array(seqs[q]); x = (np.array([FR["test"][i]["seconds"] for i in idx]) - FR["test"][idx[0]]["seconds"]) / 60
fig, a = plt.subplots(figsize=(13, 3.6))
for i, f in enumerate(sf.FEATURES):
    a.plot(x, F["test"][idx, i], lw=0.8, alpha=0.8, label=sf.SHORT[f])
dq, _ = sf.weighted_score(F["test"][idx], W_D); a.plot(x, dq, "k", lw=2, label="D (tuned)")
fx = np.array([FR["test"][i]["fixable"] for i in idx]); a.scatter(x[fx > 0], np.full((fx > 0).sum(), 1.03), marker="v", color="r", label="frame with fixable slots")
r0 = FR["test"][idx[0]]
a.set(xlabel="minutes", ylim=(-0.02, 1.08), title="test | {} | {} | D components".format(r0["camera"], r0["climate"])); a.legend(fontsize=7, ncol=8, loc="lower left")
plt.tight_layout(); plt.savefig(str(OUT / "d_timeline{}.png".format(SUF)), dpi=110); plt.show()

# risk of the slot with the most reuse errors in that sequence
fs = SL["test"]["frame"]; sel = np.isin(fs, idx)
sids = SL["test"]["slot_id"][sel]; errs = SL["test"]["target"][sel]
if len(sids):
    worst = max(set(sids), key=lambda s: errs[sids == s].sum())
    m = sel & (SL["test"]["slot_id"] == worst)
    rs, _ = sf.weighted_score(r_matrix("test")[m], W_R)
    xx = (np.array([FR["test"][f]["seconds"] for f in fs[m]]) - FR["test"][idx[0]]["seconds"]) / 60
    fig, a = plt.subplots(figsize=(13, 3))
    for i, c in enumerate(rk.COMPONENTS):
        a.plot(xx, r_matrix("test")[m][:, i], lw=0.8, alpha=0.8, label=rk.SHORT[c])
    a.plot(xx, rs, "k", lw=2, label="R (tuned)")
    e = SL["test"]["target"][m] > 0; a.scatter(xx[e], np.full(e.sum(), 1.03), marker="v", color="r", label="reuse would be wrong")
    a.set(xlabel="minutes", ylim=(-0.02, 1.08), title="test | {} | slot risk components".format(worst)); a.legend(fontsize=7, ncol=7, loc="lower left")
    plt.tight_layout(); plt.savefig(str(OUT / "r_timeline{}.png".format(SUF)), dpi=110); plt.show()

In [ ]:
# Finish: environment, status
from src.common.env_record import save_environment
save_environment(str(OUT / "environment.json"))
write_status(STATUS, "09_scene_difficulty_and_slot_risk", RUN_ID, "COMPLETE", CFG_HASH, "smoke={}".format(SMOKE),
             extra={"D_weights": RECORD["difficulty"]["weights"], "R_weights": RECORD["slot_risk"]["weights"], "aggregation": CHOSEN_AGG,
                    "test_D_capture_auc": TEST["D"]["tuned"]["capture_auc"], "test_R_slot_auc": TEST["R"]["tuned"]})
print("COMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it, or paste: the replay cell (frames, availability, fixable counts), the D table, the R table and aggregation, the TEST tables, and the three plots.
`artifacts/scene_risk/` (weights, scales, per-frame features) is the input for notebook 10, so **Save Version** with outputs.